# Prediction des matchs - Coupe du Monde 2026

Notebook de base pour recuperer des donnees football, preparer les features, puis entrainer un modele de prediction.

## 1) Setup et imports
Assure-toi d'avoir une variable `API_FOOTBALL_KEY` dans ton fichier `.env`.

In [12]:
import os
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_FOOTBALL_KEY")

if API_KEY:
    print("API key detectee.")
else:
    print("API key manquante: ajoute API_FOOTBALL_KEY dans .env")

API key detectee.


## 2) Donnees du jour depuis API-FOOTBALL
Cette cellule appelle l API pour recuperer les matchs de la date du jour (UTC).

In [13]:
from datetime import datetime, timezone

FINISHED_STATUSES = {"FT", "AET", "PEN"}
FIXTURE_COLUMNS = [
    "Match_ID",
    "Date",
    "Timestamp",
    "Status",
    "League",
    "LeagueId",
    "Season",
    "Home",
    "Away",
    "HomeTeamId",
    "AwayTeamId",
    "Venue",
    "HomeGoals",
    "AwayGoals",
    "HomeGoalsRaw",
    "AwayGoalsRaw",
    "HT_Home",
    "HT_Away",
    "IsFinished",
]

def fetch_fixtures(params: dict, api_key: str) -> list[dict]:
    if not api_key:
        raise ValueError("API_FOOTBALL_KEY est manquante. Ajoute-la dans .env.")

    url = "https://v3.football.api-sports.io/fixtures"
    headers = {"x-apisports-key": api_key}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    return payload.get("response", [])

def fixtures_to_dataframe(fixtures: list[dict]) -> pd.DataFrame:
    rows = []
    for item in fixtures:
        fixture = item.get("fixture", {})
        league = item.get("league", {})
        teams = item.get("teams", {})
        goals = item.get("goals", {})
        score = item.get("score", {})
        venue = fixture.get("venue") or {}

        home_goals = goals.get("home")
        away_goals = goals.get("away")
        status = (fixture.get("status") or {}).get("short")

        rows.append(
            {
                "Match_ID": fixture.get("id"),
                "Date": fixture.get("date"),
                "Timestamp": fixture.get("timestamp"),
                "Status": status,
                "League": league.get("name"),
                "LeagueId": league.get("id"),
                "Season": league.get("season"),
                "Home": (teams.get("home") or {}).get("name"),
                "Away": (teams.get("away") or {}).get("name"),
                "HomeTeamId": (teams.get("home") or {}).get("id"),
                "AwayTeamId": (teams.get("away") or {}).get("id"),
                "Venue": venue.get("name"),
                "HomeGoals": home_goals if home_goals is not None else 0,
                "AwayGoals": away_goals if away_goals is not None else 0,
                "HomeGoalsRaw": home_goals,
                "AwayGoalsRaw": away_goals,
                "HT_Home": (score.get("halftime") or {}).get("home"),
                "HT_Away": (score.get("halftime") or {}).get("away"),
                "IsFinished": status in FINISHED_STATUSES,
            }
        )

    frame = pd.DataFrame(rows, columns=FIXTURE_COLUMNS)
    if not frame.empty:
        frame["Date"] = pd.to_datetime(frame["Date"], utc=True, errors="coerce")

    return frame

def fetch_match_data(date: str, api_key: str) -> pd.DataFrame:
    return fixtures_to_dataframe(fetch_fixtures({"date": date}, api_key))

def fetch_team_recent_matches(team_id: int, api_key: str, last_matches: int = 8) -> pd.DataFrame:
    params = {
        "team": team_id,
        "last": last_matches,
        "status": "FT",
    }
    return fixtures_to_dataframe(fetch_fixtures(params, api_key))

today_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d")
df_all = fetch_match_data(today_utc, API_KEY)

# Garde uniquement les matchs de la competition "World Cup"
df = df_all[df_all["League"].fillna("").str.casefold().eq("world cup")].reset_index(drop=True)

print(f"Matchs World Cup recuperes pour {today_utc}: {len(df)} / {len(df_all)}")
df[["Date", "Status", "Home", "Away", "Venue"]].head(20)

Matchs World Cup recuperes pour 2026-06-15: 4 / 71


,Date,Status,Home,Away,Venue
0,2026-06-15 02:00:00+00:00,FT,Sweden,Tunisia,Estadio BBVA
1,2026-06-15 16:00:00+00:00,NS,Spain,Cape Verde Islands,Mercedes-Benz Stadium
2,2026-06-15 19:00:00+00:00,NS,Belgium,Egypt,Lumen Field
3,2026-06-15 22:00:00+00:00,NS,Saudi Arabia,Uruguay,Hard Rock Stadium


## 3) Algorithme de prediction base sur la forme recente
On calcule un score avant match a partir des derniers matchs disponibles de chaque equipe : points pris, difference de buts, efficacite offensive, solidite defensive et temps de repos.
Cette approche evite d utiliser le score final du match a predire, ce qui etait le principal probleme de la version precedente.

In [14]:
FORM_MATCHES = 8
MIN_HISTORY_MATCHES = 3
PREDICTABLE_STATUSES = {"NS", "TBD"}

def summarize_team_form(history: pd.DataFrame, team_name: str) -> dict:
    required_columns = {
        "Home",
        "Away",
        "IsFinished",
        "Date",
        "HomeGoals",
        "AwayGoals",
    }
    if history.empty or not required_columns.issubset(history.columns):
        return {
            "matches_used": 0,
            "points_per_match": 0.0,
            "goal_diff_avg": 0.0,
            "goals_for_avg": 0.0,
            "goals_against_avg": 0.0,
            "clean_sheet_rate": 0.0,
            "rest_days": np.nan,
            "form_score": 0.0,
        }

    team_history = history[
        ((history["Home"] == team_name) | (history["Away"] == team_name)) & history["IsFinished"]
    ].copy()

    if team_history.empty:
        return {
            "matches_used": 0,
            "points_per_match": 0.0,
            "goal_diff_avg": 0.0,
            "goals_for_avg": 0.0,
            "goals_against_avg": 0.0,
            "clean_sheet_rate": 0.0,
            "rest_days": np.nan,
            "form_score": 0.0,
        }

    team_history = team_history.sort_values("Date", ascending=False).reset_index(drop=True)

    goals_for = []
    goals_against = []
    points = []
    clean_sheets = []

    for row in team_history.itertuples(index=False):
        is_home_team = row.Home == team_name
        gf = row.HomeGoals if is_home_team else row.AwayGoals
        ga = row.AwayGoals if is_home_team else row.HomeGoals

        goals_for.append(gf)
        goals_against.append(ga)
        clean_sheets.append(int(ga == 0))

        if gf > ga:
            points.append(3)
        elif gf == ga:
            points.append(1)
        else:
            points.append(0)

    rest_days = np.nan
    if len(team_history) >= 2:
        latest_match = team_history.loc[0, "Date"]
        previous_match = team_history.loc[1, "Date"]
        if pd.notna(latest_match) and pd.notna(previous_match):
            rest_days = (latest_match - previous_match).days

    points_per_match = float(np.mean(points))
    goal_diff_avg = float(np.mean(np.array(goals_for) - np.array(goals_against)))
    goals_for_avg = float(np.mean(goals_for))
    goals_against_avg = float(np.mean(goals_against))
    clean_sheet_rate = float(np.mean(clean_sheets))

    form_score = (
        1.4 * points_per_match
        + 0.8 * goal_diff_avg
        + 0.3 * goals_for_avg
        - 0.2 * goals_against_avg
        + 0.2 * clean_sheet_rate
    )

    return {
        "matches_used": len(team_history),
        "points_per_match": points_per_match,
        "goal_diff_avg": goal_diff_avg,
        "goals_for_avg": goals_for_avg,
        "goals_against_avg": goals_against_avg,
        "clean_sheet_rate": clean_sheet_rate,
        "rest_days": rest_days,
        "form_score": form_score,
    }

def to_three_way_probabilities(score_gap: float) -> tuple[float, float, float]:
    raw_home = float(1 / (1 + np.exp(-score_gap)))
    raw_away = float(1 / (1 + np.exp(score_gap)))
    raw_draw = float(np.clip(0.30 - 0.12 * abs(score_gap), 0.12, 0.30))

    total = raw_home + raw_draw + raw_away
    return raw_home / total, raw_draw / total, raw_away / total

fixtures_to_predict = df[df["Status"].isin(PREDICTABLE_STATUSES)].copy()
history_cache = {}
prediction_rows = []

for match in fixtures_to_predict.itertuples(index=False):
    if pd.isna(match.HomeTeamId) or pd.isna(match.AwayTeamId):
        continue

    home_team_id = int(match.HomeTeamId)
    away_team_id = int(match.AwayTeamId)

    if home_team_id not in history_cache:
        history_cache[home_team_id] = fetch_team_recent_matches(home_team_id, API_KEY, FORM_MATCHES)
    if away_team_id not in history_cache:
        history_cache[away_team_id] = fetch_team_recent_matches(away_team_id, API_KEY, FORM_MATCHES)

    home_form = summarize_team_form(history_cache[home_team_id], match.Home)
    away_form = summarize_team_form(history_cache[away_team_id], match.Away)

    score_gap = (
        1.20 * (home_form["points_per_match"] - away_form["points_per_match"])
        + 0.90 * (home_form["goal_diff_avg"] - away_form["goal_diff_avg"])
        + 0.35 * (home_form["goals_for_avg"] - away_form["goals_for_avg"])
        + 0.25 * (away_form["goals_against_avg"] - home_form["goals_against_avg"])
        + 0.15 * (home_form["clean_sheet_rate"] - away_form["clean_sheet_rate"])
    )

    if not np.isnan(home_form["rest_days"]) and not np.isnan(away_form["rest_days"]):
        score_gap += 0.03 * (home_form["rest_days"] - away_form["rest_days"])

    home_prob, draw_prob, away_prob = to_three_way_probabilities(score_gap)

    prediction_map = {
        "Home": home_prob,
        "Draw": draw_prob,
        "Away": away_prob,
    }
    predicted_side = max(prediction_map, key=prediction_map.get)
    label_map = {
        "Home": match.Home,
        "Draw": "Match nul",
        "Away": match.Away,
    }

    prediction_rows.append(
        {
            "Date": match.Date,
            "Home": match.Home,
            "Away": match.Away,
            "Prediction": label_map[predicted_side],
            "HomeWinProb": round(home_prob, 3),
            "DrawProb": round(draw_prob, 3),
            "AwayWinProb": round(away_prob, 3),
            "HomeFormScore": round(home_form["form_score"], 2),
            "AwayFormScore": round(away_form["form_score"], 2),
            "HomeMatchesUsed": home_form["matches_used"],
            "AwayMatchesUsed": away_form["matches_used"],
            "HistoryNote": "ok" if min(home_form["matches_used"], away_form["matches_used"]) >= MIN_HISTORY_MATCHES else "historique limite",
        }
    )

predictions_df = pd.DataFrame(prediction_rows)
predictions_df

,Date,Home,Away,Prediction,HomeWinProb,DrawProb,AwayWinProb,HomeFormScore,AwayFormScore,HomeMatchesUsed,AwayMatchesUsed,HistoryNote
0,2026-06-15 16:00:00+00:00,Spain,Cape Verde Islands,Spain,0.385,0.231,0.385,0.0,0.0,0,0,historique limite
1,2026-06-15 19:00:00+00:00,Belgium,Egypt,Belgium,0.385,0.231,0.385,0.0,0.0,0,0,historique limite
2,2026-06-15 22:00:00+00:00,Saudi Arabia,Uruguay,Saudi Arabia,0.385,0.231,0.385,0.0,0.0,0,0,historique limite


## 4) Lecture des predictions
Le tableau ci-dessous donne une probabilite simple 1 / N / 2 issue de la forme recente.
C est un algorithme de scoring pre-match, utile pour un premier prototype avant de passer a un modele supervise entraine sur plusieurs saisons.

In [15]:
if predictions_df.empty:
    print("Aucun match exploitable pour calculer des predictions.")
else:
    columns = [
        "Date",
        "Home",
        "Away",
        "Prediction",
        "HomeWinProb",
        "DrawProb",
        "AwayWinProb",
        "HomeFormScore",
        "AwayFormScore",
        "HomeMatchesUsed",
        "AwayMatchesUsed",
        "HistoryNote",
    ]
    predictions_df[columns].sort_values(["Date", "HomeWinProb"], ascending=[True, False]).reset_index(drop=True)